# Evex Initial Data Analysis

In [9]:
import json
import locale
import pickle
from datetime import UTC, datetime, time, timedelta

import numpy as np
import pandas as pd

# from utils import fetch_all_issues, run_prompt, get_summary
import plotly.express as px
import plotly.graph_objects as go
import pytz
from dotenv import load_dotenv
from jira import JIRA
from plotly.subplots import make_subplots

%load_ext autoreload
%autoreload 2
locale.setlocale(locale.LC_TIME, "de_DE.UTF-8")
load_dotenv()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


True

In [74]:
from data_transformation import load_issues, load_issues_Amparex
from jira_loader import fetch_jira_issues

In [56]:
start_date = datetime.now(UTC) - timedelta(days=183)
end_date = datetime.now(UTC)

tz = pytz.UTC

start_dt = tz.localize(datetime.combine(start_date, time.min))
end_dt = tz.localize(datetime.combine(end_date, time.max))

In [57]:
issues_ipro = fetch_jira_issues(start_dt, end_dt, max_issues=10000, project="SDIPR")

Total issues fetched: 5741


In [88]:
issues_ax = fetch_jira_issues(start_dt, end_dt, max_issues=50000, project="SDAX")

Total issues fetched: 16393


In [90]:
df_ax = load_issues_Amparex(issues_ax)

In [93]:
df_ax.to_csv("jira_tickets_ax.csv", sep=";", index=False, encoding="utf-8")

In [78]:
df_ax.columns

Index(['firma', 'key', 'summary', 'description', 'status', 'status_category',
       'created', 'updated', 'labels', 'source', 'priority', 'category',
       'issuetype', 'main_category_id', 'sub_category_id',
       'currentstatus_name', 'currentstatus_date', 'comments', 'request_type',
       'clones', 'clone_types', 'cloned_by', 'n_clones', 'zentrale', 'filiale',
       'Link', 'clones_of_clones', 'clone_types_of_clones', 'is_done',
       'resolved_at', 'time_to_resolution_h',
       'time_to_resolution_calendar_days', 'resolution',
       'time_to_resolution_biz_hours', 'time_to_resolution_biz_days',
       'time_to_resolution_bin', 'business_days_created_to_updated',
       'business_days_created_to_resolved', 'created_string', 'updated_string',
       'year', 'month', 'week_number', 'week_string', 'month_string',
       'Hauptkategorie', 'Unterkategorie', 'clone_in_project',
       'has_exax_clone', 'has_axt_clone_clone'],
      dtype='object')

In [79]:
df_ax.issuetype.value_counts()

issuetype
Allgemeine Anfrage    100
Name: count, dtype: int64

In [86]:
df_ax["comments"] = ""

for i in range(len(df_ax)):
    try:
        df_ax.loc[i, "comments"] = str(issues_ax[i]["fields"]["comment"])
    except Exception:  # noqa: BLE001 - deliberate catch-all in exploratory code
        print(i)
        df_ax.loc[i, "comments"] = ""

In [87]:
df_ax[df_ax["comments"] != ""]

,firma,key,summary,description,status,status_category,created,updated,labels,source,...,year,month,week_number,week_string,month_string,Hauptkategorie,Unterkategorie,clone_in_project,has_exax_clone,has_axt_clone_clone
0,Amparex,SDAX-407427,Dokument Rechnung QR geändert,hörenhoch3 weg und nur Jaqueline Schröppel mit...,Fertig,Fertig,2026-04-27 17:07:56.602000+02:00,2026-04-27 17:08:03.041000+02:00,[],Anruf,...,2026,4,18,2026-W18,2026-04,Stammdaten,Dokumentvorlagen,-,False,False
1,Amparex,SDAX-407426,Einrichtung Buchhaltung,"Buha noch nicht eingerichtet, wünscht AN\n\nev...",Frage an Fachabteilung,In Arbeit,2026-04-27 17:01:29.527000+02:00,2026-04-27 17:03:47.935000+02:00,[],Anruf,...,2026,4,18,2026-W18,2026-04,Buchhaltung,NA,EXAX,True,False
2,Amparex,SDAX-407425,HG aus rep wieder einlagern,Hub Issue From Search - 004978195576611,Fertig,Fertig,2026-04-27 16:59:16.713000+02:00,2026-04-27 16:59:23.341000+02:00,[],Anruf,...,2026,4,18,2026-W18,2026-04,Lagerverwaltung,NA,-,False,False
3,Amparex,SDAX-407424,Braucht Hilfe bei Serviceverträgen,Findet keine Bestätigung für die Abbuchung.\n\...,Warten auf Support,Zu erledigen,2026-04-27 16:53:20.664000+02:00,2026-04-27 16:53:49.919000+02:00,[],Anruf,...,2026,4,18,2026-W18,2026-04,Serviceverträge und Rechnungswesen,NA,-,False,False
4,Amparex,SDAX-407423,LibreOffice ohne Funktion,None,Fertig,Fertig,2026-04-27 16:50:17.438000+02:00,2026-04-27 16:50:23.863000+02:00,[],Anruf,...,2026,4,18,2026-W18,2026-04,Libre Office,NA,-,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Amparex,SDAX-407332,Buha export,None,Fertig,Fertig,2026-04-27 11:48:10.313000+02:00,2026-04-27 11:48:16.258000+02:00,[],Anruf,...,2026,4,18,2026-W18,2026-04,Buchhaltung,NA,-,False,False
96,Amparex,SDAX-407331,fortlaufende Nummer für dokument,nochmal zu [[SDAX-407323] fortlaufende Nummer ...,Fertig,Fertig,2026-04-27 11:44:26.458000+02:00,2026-04-27 11:44:32.314000+02:00,[],Anruf,...,2026,4,18,2026-W18,2026-04,Stammdaten,Dokumentvorlagen,-,False,False
97,Amparex,SDAX-407330,Rückfrage Ec Zahlung erneut verbuchen,None,Fertig,Fertig,2026-04-27 11:43:22.149000+02:00,2026-04-27 11:43:28.206000+02:00,[],Anruf,...,2026,4,18,2026-W18,2026-04,Kasse,NA,-,False,False
98,Amparex,SDAX-407329,FolgeReppauschale anfordern,Academyfilm gezeigt und auch manuell,Fertig,Fertig,2026-04-27 11:41:55.786000+02:00,2026-04-27 11:42:03.633000+02:00,[],Anruf,...,2026,4,18,2026-W18,2026-04,Abrechnung Kostenträger,NA,-,False,False


In [58]:
issues_ipro[0]

{'id': '477435',
 'self': 'https://amparex.atlassian.net/rest/api/2/issue/477435',
 'key': 'SDIPR-100012795',
 'fields': {'statuscategorychangedate': '2026-04-27T17:36:10.080+0200',
  'customfield_10070': None,
  'customfield_10071': None,
  'customfield_10072': None,
  'customfield_10073': None,
  'customfield_10074': None,
  'customfield_10075': None,
  'customfield_10076': None,
  'fixVersions': [],
  'statusCategory': {'self': 'https://amparex.atlassian.net/rest/api/2/statuscategory/2',
   'id': 2,
   'key': 'new',
   'colorName': 'blue-gray',
   'name': 'Zu erledigen'},
  'customfield_11564': None,
  'resolution': None,
  'customfield_10905': None,
  'lastViewed': None,
  'customfield_10060': None,
  'customfield_10061': None,
  'customfield_10062': None,
  'customfield_10065': None,
  'customfield_12004': None,
  'priority': {'self': 'https://amparex.atlassian.net/rest/api/2/priority/10001',
   'iconUrl': 'https://amparex.atlassian.net/images/icons/priorities/medium.svg',
   'nam

In [59]:
df_new_ipro = load_issues(issues_ipro)

In [60]:
df_new_ipro["reporter"] = ""
for i in range(len(df_new_ipro)):
    try:
        df_new_ipro.loc[i, "reporter"] = issues_ipro[i]["fields"]["reporter"][
            "emailAddress"
        ]
    except Exception:  # noqa: BLE001 - deliberate catch-all in exploratory code
        df_new_ipro.loc[i, "reporter"] = ""

In [61]:
df_new_ipro.head()

,firma,key,summary,description,status,status_category,created,updated,labels,source,...,month,week_number,week_string,month_string,Hauptkategorie,Unterkategorie,clone_in_project,has_exax_clone,has_axt_clone_clone,reporter
0,IPRO,SDIPR-100012795,Probleme mit dem Import von IPRO an Winfit,"Frau Lieb via FL:\n\nHat das Problem, dass wen...",Warten auf Support,Zu erledigen,2026-04-27 17:36:09.404000+02:00,2026-04-27 17:36:15.254000+02:00,[],Anruf,...,4,18,2026-W18,2026-04,SST Industrieprogramme,Rodenstock WinFit,-,False,,
1,IPRO,SDIPR-100012792,Fassungshersteller Licefa,"Hr.Gahbauer über FL:\n\nKatalog neu zuordnen, ...",Fertig,Fertig,2026-04-27 16:56:32.684000+02:00,2026-04-27 16:57:00.359000+02:00,[],Anruf,...,4,18,2026-W18,2026-04,Datenservice/OLDS,Fassungs-Kataloge,-,False,,505585@evex-group.com
2,IPRO,SDIPR-100012791,Filialvernetzungsbenachrichtigung (11) von Op...,Von: hassfurt.nohe@schaefer-nohe.de [hassfurt....,Warten auf Support,Zu erledigen,2026-04-27 16:35:26.803000+02:00,2026-04-27 16:37:15.946000+02:00,[],Anruf,...,4,18,2026-W18,2026-04,Filialvernetzung,NA,-,False,,507118@evex-group.com
3,IPRO,SDIPR-100012790,Hoya GT-3000 anschließen [Ticket#2026042750004...,"Herr Tim Bernhardt über FL,\n\nmöchte in Lohfe...",Warten auf Support,Zu erledigen,2026-04-27 16:08:11.941000+02:00,2026-04-27 16:55:31.476000+02:00,[],Anruf,...,4,18,2026-W18,2026-04,SST Werkstatt,Hoya,-,False,,
4,IPRO,SDIPR-100012789,Layout Computerkasse,"Herr Heimers via FL:\n\nHat das Problem, dass ...",Fertig,Fertig,2026-04-27 15:58:55.205000+02:00,2026-04-27 15:59:11.285000+02:00,[],Anruf,...,4,18,2026-W18,2026-04,Computerkasse,NA,-,False,,505047@evex-group.com


In [72]:
df_new_ipro.source.value_counts()

source
Anruf           5609
Portal           118
E-Mail            13
KI-Assistent       1
Name: count, dtype: int64

In [62]:
# load json file
with open("jira_user_org_mapping.json", "r") as f:
    jira_user_mapping = json.load(f)

# turn jira_user_mapping into dataframe
df_jira_user_mapping = pd.DataFrame(jira_user_mapping)
df_jira_user_mapping.head()

,organizationId,organizationName,accountId,displayName,emailAddress
0,8715,Hörland e.K.-505391-17151-Wunsiedler Str. 59-9...,qm:e8aa04e1-399f-41c0-9191-6f94b0bd264e:ed309f...,Default Ansprechpartner 505391,505391@evex-group.com
1,8171,Optiker Olbrich-504848-15793-Tevesstraße 37-78...,qm:e8aa04e1-399f-41c0-9191-6f94b0bd264e:5b68f6...,Inh. Dieter Olbrich,optikerolbrich.blumberg@t-online.de
2,8171,Optiker Olbrich-504848-15793-Tevesstraße 37-78...,qm:e8aa04e1-399f-41c0-9191-6f94b0bd264e:be4d49...,Default Ansprechpartner 504848,504848@evex-group.com
3,7732,Alo Kramer Augenoptik & Hörgeräte-504405-14696...,qm:e8aa04e1-399f-41c0-9191-6f94b0bd264e:1ab01f...,Default Ansprechpartner 504405,504405@evex-group.com
4,7732,Alo Kramer Augenoptik & Hörgeräte-504405-14696...,qm:e8aa04e1-399f-41c0-9191-6f94b0bd264e:fd3e1c...,Frau Petra Grützmacher,alo.kramer@t-online.de


In [63]:
df_new_ipro = df_new_ipro.merge(
    df_jira_user_mapping, left_on="reporter", right_on="emailAddress", how="left"
)

In [64]:
df_new_ipro["Kunde"] = df_new_ipro["organizationName"].str.extract(r"^(.*?)-\d+-")

In [65]:
df_new_ipro.head()

,firma,key,summary,description,status,status_category,created,updated,labels,source,...,clone_in_project,has_exax_clone,has_axt_clone_clone,reporter,organizationId,organizationName,accountId,displayName,emailAddress,Kunde
0,IPRO,SDIPR-100012795,Probleme mit dem Import von IPRO an Winfit,"Frau Lieb via FL:\n\nHat das Problem, dass wen...",Warten auf Support,Zu erledigen,2026-04-27 17:36:09.404000+02:00,2026-04-27 17:36:15.254000+02:00,[],Anruf,...,-,False,,,NaN,NaN,NaN,NaN,NaN,NaN
1,IPRO,SDIPR-100012792,Fassungshersteller Licefa,"Hr.Gahbauer über FL:\n\nKatalog neu zuordnen, ...",Fertig,Fertig,2026-04-27 16:56:32.684000+02:00,2026-04-27 16:57:00.359000+02:00,[],Anruf,...,-,False,,505585@evex-group.com,8911,OPTIK GAHBAUER-505585-17609-Marktplatz 31-8557...,qm:e8aa04e1-399f-41c0-9191-6f94b0bd264e:7f157f...,Default Ansprechpartner 505585,505585@evex-group.com,OPTIK GAHBAUER
2,IPRO,SDIPR-100012791,Filialvernetzungsbenachrichtigung (11) von Op...,Von: hassfurt.nohe@schaefer-nohe.de [hassfurt....,Warten auf Support,Zu erledigen,2026-04-27 16:35:26.803000+02:00,2026-04-27 16:37:15.946000+02:00,[],Anruf,...,-,False,,507118@evex-group.com,13654,Optik Nohe-507118-16015-Hauptstr. 48-97437 Haß...,qm:e8aa04e1-399f-41c0-9191-6f94b0bd264e:8d6106...,Default Ansprechpartner 507118,507118@evex-group.com,Optik Nohe
3,IPRO,SDIPR-100012790,Hoya GT-3000 anschließen [Ticket#2026042750004...,"Herr Tim Bernhardt über FL,\n\nmöchte in Lohfe...",Warten auf Support,Zu erledigen,2026-04-27 16:08:11.941000+02:00,2026-04-27 16:55:31.476000+02:00,[],Anruf,...,-,False,,,NaN,NaN,NaN,NaN,NaN,NaN
4,IPRO,SDIPR-100012789,Layout Computerkasse,"Herr Heimers via FL:\n\nHat das Problem, dass ...",Fertig,Fertig,2026-04-27 15:58:55.205000+02:00,2026-04-27 15:59:11.285000+02:00,[],Anruf,...,-,False,,505047@evex-group.com,8372,Heimers Sehen Verstehen-505047-16294-Katzwange...,qm:e8aa04e1-399f-41c0-9191-6f94b0bd264e:2b4f37...,Default Ansprechpartner 505047,505047@evex-group.com,Heimers Sehen Verstehen


In [70]:
df_new_ipro.to_csv("jira_tickets_ipro.csv", sep=";", index=False, encoding="utf-8-sig")

In [66]:
df_new_ipro.Kunde.value_counts()

Kunde
OUNDA GmbH                           244
Neusehland Hartmann GmbH & Co. KG     89
Matthias Kaulard GmbH & Co.KG         36
GRONDE sehen & hören GmbH             35
Binder-Optik GmbH                     35
                                    ... 
Holzmann Optik                         1
Optik BRILLIG e.U.                     1
Augenoptik Kalweit                     1
Oehm Optik                             1
Optik Eichhammer                       1
Name: count, Length: 1206, dtype: int64

In [20]:
df_new_ipro[["zentrale", "filiale"]]

,zentrale,filiale
0,ID_9751,
1,ID_10227,ID_58934
2,ID_9717,ID_222031
3,ID_8872,
4,ID_9923,
...,...,...
95,ID_9373,
96,ID_8420,
97,ID_8338,
98,ID_10078,


In [23]:
df_new_ipro.zentrale.value_counts()

zentrale
ID_9785     6
ID_9717     3
ID_8789     3
ID_8407     2
ID_10515    2
           ..
ID_8913     1
ID_8011     1
ID_10227    1
ID_9898     1
ID_9716     1
Name: count, Length: 81, dtype: int64

In [27]:
df_new_ipro[df_new_ipro["zentrale"] == "ID_9785"][["zentrale", "filiale"]]

,zentrale,filiale
22,ID_9785,
30,ID_9785,ID_17250
40,ID_9785,ID_17264
59,ID_9785,
71,ID_9785,ID_17358
76,ID_9785,ID_17305


## Pull Issue list

In [ ]:
# Create a Jira client and authenticate with API key
# basic_auth = ("bl@flex.capital", "...")
basic_auth = "..."

jira = JIRA(server="https://amparex.atlassian.net/", basic_auth=basic_auth)

In [6]:
jira.projects()

[]

In [15]:
jql = "project = EXIPR ORDER BY created DESC"
page = jira.enhanced_search_issues(
    jql_str=jql,
    maxResults=20,  # per API call
    json_result=True,
)
page

{'issues': [], 'isLast': True}

In [13]:
# Disabled: fetch_all_issues() lives in utils.py, which is not part of this
# repository. Restore utils.py to re-enable this live Jira pull.
# jql = "project = EXIPR ORDER BY created DESC"
# issues = fetch_all_issues(jira, jql, max_issues=5000)

Total issues fetched: 0


In [11]:
# Disabled: fetch_all_issues() lives in utils.py, which is not part of this
# repository, so the live Jira pull below cannot run. The cached pickle is
# loaded instead. Restore utils.py to re-enable the pull.
#
# run_jira_pull = True
# if run_jira_pull:
#     jql = "project = EXIPR ORDER BY created DESC"
#     issues = fetch_all_issues(jira, jql, max_issues=5000)
#     with open("issues.pkl", "wb") as f:
#         pickle.dump(issues, f)

# retrieve issues from pickle file
with open("issues.pkl", "rb") as f:
    issues = pickle.load(f)

Total issues fetched: 0


In [12]:
len(issues)

0

In [7]:
issues[0]["fields"]

{'statuscategorychangedate': '2025-12-01T16:12:45.571+0100',
 'customfield_10070': None,
 'customfield_10071': None,
 'customfield_10072': None,
 'customfield_10073': None,
 'customfield_10074': None,
 'customfield_10075': None,
 'customfield_10076': None,
 'fixVersions': [],
 'statusCategory': {'self': 'https://amparex.atlassian.net/rest/api/2/statuscategory/3',
  'id': 3,
  'key': 'done',
  'colorName': 'green',
  'name': 'Fertig'},
 'customfield_11564': None,
 'resolution': {'self': 'https://amparex.atlassian.net/rest/api/2/resolution/10006',
  'id': '10006',
  'description': 'Die Arbeit für diesen Vorgang wurde abgeschlossen.',
  'name': 'Fertig'},
 'customfield_10905': None,
 'lastViewed': None,
 'customfield_10060': None,
 'customfield_10061': None,
 'customfield_10062': None,
 'customfield_10065': None,
 'customfield_12004': None,
 'priority': {'self': 'https://amparex.atlassian.net/rest/api/2/priority/10001',
  'iconUrl': 'https://amparex.atlassian.net/images/icons/priorities/m

In [18]:
# read json file "fields.json"
with open("response_ticket.json", "r") as f:
    fields = json.load(f)

# save back with indent 4
with open("response_ticket.json", "w") as f:
    json.dump(fields, f, indent=4)

In [8]:
# read json from data/jira-servicedesk-schema-objects.json
with open("data/jira-servicedesk-schema-objects.json", "r") as f:
    schema = json.load(f)

object_id_to_name = {v["id"]: v["name"] for v in schema["values"]}

In [32]:
df = {
    "key": [],
    "summary": [],
    "description": [],
    "status": [],
    "status_category": [],
    "created": [],
    "updated": [],
    "labels": [],
    "source": [],
    "priority": [],
    "category": [],
    "issuetype": [],
    "main_category_id": [],
    "sub_category_id": [],
    "currentstatus_name": [],
    "currentstatus_date": [],
    "comments": [],
    "request_type": [],
}
for issue in issues:
    df["key"].append(issue["key"])
    df["summary"].append(issue["fields"]["summary"])
    df["description"].append(issue["fields"]["description"])
    df["status"].append(issue["fields"]["status"]["name"])
    df["status_category"].append(issue["fields"]["status"]["statusCategory"]["name"])
    # df['creator'].append(issue['fields']['creator']['displayName'])
    df["issuetype"].append(issue["fields"]["issuetype"]["name"])
    df["created"].append(issue["fields"]["created"])
    df["updated"].append(issue["fields"]["updated"])
    df["labels"].append(issue["fields"]["labels"])
    df["priority"].append(issue["fields"]["priority"]["name"])
    df["category"].append(issue["fields"]["customfield_10065"])

    if issue["fields"]["customfield_10010"] is not None:
        df["request_type"].append(
            issue["fields"]["customfield_10010"]["requestType"]["name"]
        )
    else:
        df["request_type"].append("")
    if issue["fields"]["comment"] is not None:
        df["comments"].append(
            "\n\n".join([c["body"] for c in issue["fields"]["comment"]["comments"]])
        )
    else:
        df["comments"].append([])

    try:
        df["currentstatus_name"].append(
            issue["fields"]["customfield_10010"]["currentStatus"]["status"]
        )
        df["currentstatus_date"].append(
            issue["fields"]["customfield_10010"]["currentStatus"]["statusDate"]["jira"]
        )
    except Exception:  # noqa: BLE001 - deliberate catch-all in exploratory code
        df["currentstatus_name"].append("")
        df["currentstatus_date"].append("")

    try:
        v = issue["fields"]["customfield_10675"]["value"]
        df["source"].append(v)
    except Exception:  # noqa: BLE001 - deliberate catch-all in exploratory code
        df["source"].append("")

    cf = issue["fields"]["customfield_10680"]
    if len(cf) > 0:
        df["main_category_id"].append(cf[0]["objectId"])
    else:
        df["main_category_id"].append("")
    cf = issue["fields"]["customfield_10679"]
    if len(cf) > 0:
        df["sub_category_id"].append(cf[0]["objectId"])
    else:
        df["sub_category_id"].append("")


df = pd.DataFrame(df)
### convert created, updated to datetime
df["created"] = pd.to_datetime(df["created"], errors="coerce", utc=True)
df["updated"] = pd.to_datetime(df["updated"], errors="coerce", utc=True)
df["currentstatus_date"] = pd.to_datetime(
    df["currentstatus_date"], errors="coerce", utc=True
)
df["time_to_resolution_h"] = (
    df["currentstatus_date"] - df["created"]
).dt.total_seconds() / 3600
df["time_to_resolution_days"] = (df["currentstatus_date"] - df["created"]).dt.days
df["resolution"] = "> 1 day"
df["resolution"] = np.where(df["time_to_resolution_days"] <= 1, "Same day", "> 1 day")
df["bdays"] = np.busday_count(
    df["created"].to_numpy(dtype="datetime64[D]"),
    df["updated"].to_numpy(dtype="datetime64[D]"),
)
df["created_string"] = df["created"].dt.strftime("%Y-%m-%d")
df["updated_string"] = df["updated"].dt.strftime("%Y-%m-%d")
df["year"] = df["created"].dt.year
df["month"] = df["created"].dt.month
df["Hauptkategorie"] = df["main_category_id"].map(object_id_to_name)
df["Unterkategorie"] = df["sub_category_id"].map(object_id_to_name)
# put time to resolution into bins
bins = [0, 1, 2, 4, 8, 24, 48, 72, 7 * 24, 14 * 24, 21 * 24]
df["time_to_resolution_bin"] = pd.cut(df["time_to_resolution_h"], bins=bins)
df["time_to_resolution_bin"] = df["time_to_resolution_bin"].apply(
    lambda x: f"{int(x.left)}–{int(x.right)}"
)
df.head(5)

,key,summary,description,status,status_category,created,updated,labels,source,priority,...,time_to_resolution_days,resolution,bdays,created_string,updated_string,year,month,Hauptkategorie,Unterkategorie,time_to_resolution_bin
0,SDIPR-100008182,CoKa falsche Buchung,Kundin hat am Freitag zu viel in einem Auftrag...,Fertig,Fertig,2025-12-01 15:12:14.587000+00:00,2025-12-01 15:13:20.526000+00:00,[],Anruf,Normal,...,0.0,Same day,0,2025-12-01,2025-12-01,2025,12,Computerkasse,NaN,0–1
1,SDIPR-100008181,Datensicherung lief nicht,"Sticks sahen gut aus, Server lief seit 60 Tage...",Fertig,Fertig,2025-12-01 14:39:41.540000+00:00,2025-12-01 14:40:39.269000+00:00,[],Anruf,Normal,...,0.0,Same day,0,2025-12-01,2025-12-01,2025,12,Datensicherung,NaN,0–1
2,SDIPR-100008180,Hörgeräte erscheinen nicht in Auflistung der E...,Mail:\n\n-------- Weitergeleitete Nachricht --...,Fertig,Fertig,2025-12-01 14:37:07.047000+00:00,2025-12-01 15:20:04.191000+00:00,[],Anruf,Normal,...,0.0,Same day,0,2025-12-01,2025-12-01,2025,12,Akustik,NaN,0–1
3,SDIPR-100008179,KuWe erstellt / Angebotskennzeichen,Kunde ein Liste erstellt für sein ausgewähltes...,Fertig,Fertig,2025-12-01 14:08:28.424000+00:00,2025-12-01 14:09:17.126000+00:00,[],Anruf,Normal,...,0.0,Same day,0,2025-12-01,2025-12-01,2025,12,Kundenwerbung,NaN,0–1
4,SDIPR-100008178,Paskal - ATV wird unter Verbindungen rot angez...,Herr Klotz via FL:\n\nIn dieser Filiale wird d...,Fertig,Fertig,2025-12-01 13:54:34.101000+00:00,2025-12-01 14:38:00.133000+00:00,[],Anruf,Rot,...,0.0,Same day,0,2025-12-01,2025-12-01,2025,12,Paskal 3D,NaN,0–1


In [12]:
df.shape

(5000, 28)

In [13]:
result = (
    df[df["month"] == 11][["created_string", "key"]]
    .groupby("created_string")
    .count()
    .reset_index()
)
print(result["key"].mean())
# plot using plotly
fig = px.bar(result, x="created_string", y="key", text="key")
# set width of plot
fig.update_layout(width=1000)
# add x label
fig.update_xaxes(title_text="Date")
# add y label
fig.update_yaxes(title_text="Count of tickets")
fig.show()

46.5


In [14]:
df["request_type"].value_counts()

request_type
Allgemeine Anfrage             3509
Anfrage per E-Mail             1099
Anfrage per Voice-Mail (AB)     270
Ticket für Support (intern)     112
                                  6
Intern                            3
Info zum Kunden                   1
Name: count, dtype: int64

In [15]:
# Plot request_type in bar chart
result = df[df["request_type"] != ""]["request_type"].value_counts().reset_index()
result["count"] = result["count"] / result["count"].sum()
fig = px.bar(result, x="request_type", y="count", text="count")
# set width of plot
# add share as labels inside of bars, as percentage
fig.update_traces(
    textposition="inside", insidetextanchor="middle", texttemplate="%{text:.1%}"
)
fig.update_layout(width=1000)
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
fig.show()

## Open Ticket analysis

In [ ]:
dfopen = df[df["status_category"] != "Fertig"].copy()
todays_date = pd.Timestamp.now(tz="UTC")
dfopen["days_open"] = (todays_date - dfopen["created"]).dt.days
dfopen["weeks_open"] = -np.floor(dfopen["days_open"] / 7)
result = (
    dfopen[df["status_category"] == "In Arbeit"][
        ["weeks_open", "Hauptkategorie", "status_category"]
    ]
    .groupby(["weeks_open", "Hauptkategorie"])
    .count()
    .reset_index()
)

# plot: weeks open on x axis, count of tickets per category on y axis as stacked vertical bars
fig = px.bar(result, x="weeks_open", y="status_category", color="Hauptkategorie")
fig.update_layout(width=1000)
fig.show()

/var/folders/9v/qplwh6y13f93b4bjw7srn70r0000gn/T/ipykernel_64454/1791412562.py:5: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [31]:
dfopen = df[df["status_category"] != "Fertig"].copy()
todays_date = pd.Timestamp.now(tz="UTC")
dfopen["days_open"] = (todays_date - dfopen["created"]).dt.days
dfopen["weeks_open"] = -np.floor(dfopen["days_open"] / 7)
result = (
    dfopen[df["status_category"] == "In Arbeit"][["weeks_open", "status", "key"]]
    .groupby(["weeks_open", "status"])
    .count()
    .reset_index()
)

# plot: weeks open on x axis, count of tickets per category on y axis as stacked vertical bars
fig = px.bar(result, x="weeks_open", y="key", color="status")
fig.update_layout(width=1000)
fig.show()

/var/folders/9v/qplwh6y13f93b4bjw7srn70r0000gn/T/ipykernel_64454/1398548683.py:5: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [11]:
result = (
    df[df["status_category"] != "Fertig"][["status_category", "status", "key"]]
    .groupby(["status_category", "status"])
    .count()
    .reset_index()
    .sort_values("key", ascending=False)
)
# plot using plotly with status_category on x axis and status on y axis
fig = px.bar(result, x="status_category", y="key", color="status", text="status")
# add labels inside of bars
fig.update_traces(textposition="inside", insidetextanchor="middle")
# set width of plot
fig.update_layout(width=1000)
fig.update_layout(
    title="Status and Status Category for open tickets",
)
# make bars in shades of grey
fig.update_traces(marker_color=px.colors.qualitative.Plotly)
# add y axis label
fig.update_yaxes(title_text="Count of tickets")
# remove legend
fig.update_layout(showlegend=False, uniformtext_minsize=10)
fig.show()

In [12]:
df[df["status_category"] == "In Arbeit"]["status"].value_counts()

status
Warten auf Kunde               60
❓ Frage an Fachabteilung       27
In Arbeit                      27
Warten auf Intern              15
Brauche Info/Hilfe             10
Überwachen                      6
Warten auf Termin               5
Warten auf Lieferanten          4
❗️Antwort von Fachabteilung     4
Anfordern weiterer Infos        3
Info an Kunden                  1
Warten auf Termin mit KD        1
Name: count, dtype: int64

In [14]:
df[df["priority"].isin(["Hoch", "Sehr Hoch", "Rot"])][
    ["Hauptkategorie", "priority"]
].groupby("Hauptkategorie").count().reset_index()

,Hauptkategorie,priority
0,Akustik,3
1,Aufgabenplaner,1
2,Auftrag Allgemein,24
3,Bestellwesen CL,1
4,Bestellwesen Gläser,6
5,Business Hub,1
6,Computerkasse,22
7,Datenbank,5
8,Datenservice/OLDS,2
9,Datensicherung,8


## Time to resolution analysis

In [15]:
# plot time to resolution bin counts using plotly
# sort by midpoint of intervals/bins
result = (
    df[df["currentstatus_name"] == "Fertig"][["time_to_resolution_bin", "key"]]
    .groupby("time_to_resolution_bin")
    .count()
    .reset_index()
)
result["key"] = result["key"] / result["key"].sum()
fig = px.bar(result, x="time_to_resolution_bin", y="key")
# add x axis label

fig.update_layout(
    title="Share of tickets by time to resolution",
)
fig.update_xaxes(title_text="Time to resolution in hours")
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
# set width of plot
fig.update_layout(width=1000)
fig.show()

/var/folders/9v/qplwh6y13f93b4bjw7srn70r0000gn/T/ipykernel_34735/1899814076.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [16]:
# plot time to resolution bin counts using plotly
# sort by midpoint of intervals/bins
result = (
    df[df["currentstatus_name"] == "Fertig"][["bdays"]].value_counts().reset_index()
)
result["count"] = result["count"] / result["count"].sum()
fig = px.bar(result, x="bdays", y="count")
# add x axis label)
fig.update_layout(
    title="Share of tickets by time to resolution in business days",
)
fig.update_xaxes(title_text="Time to resolution in business days")
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
# set x axis range
fig.update_xaxes(range=[-1, 16])
fig.update_yaxes(range=[0, 0.7])
# set x tick values
fig.update_xaxes(tickvals=list(range(16)))
# set width of plot
fig.update_layout(width=1000)
fig.show()

In [43]:
# Analyze average time to resolution for top 15 main categories
# compute average time and count by main category
top15 = (
    df[df["currentstatus_name"] == "Fertig"]["Hauptkategorie"]
    .value_counts()
    .head(14)
    .index
)
df_top15 = df[df["Hauptkategorie"].isin(top15)]

result = df[
    (df["currentstatus_name"] == "Fertig") & (df["Hauptkategorie"].isin(top15))
][["Hauptkategorie", "time_to_resolution_h"]]

# group by main category and compute median time to resolution as well as count
result = (
    result.groupby("Hauptkategorie")
    .agg(
        count=("Hauptkategorie", "size"),
        median_time_delta=("time_to_resolution_h", "median"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)

# Plot the main categories with a percentage bar in terms of count and include the cumulative count as a line
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar trace
fig.add_trace(
    go.Bar(
        x=result["Hauptkategorie"],
        y=result["median_time_delta"],
        name="Median time to resolution in hours",
    ),
    secondary_y=False,
)

# Line trace (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=result["Hauptkategorie"],
        y=result["count"],
        mode="lines+markers",
        name="Count of tickets",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Median time to resolution in hours by main category (and count)",
    xaxis_title="Hauptkategorie",
    yaxis_title="Share of tickets",
)
fig.update_yaxes(title_text="Cumulative share", secondary_y=True)
# add x axis label in font size 8
fig.update_xaxes(title_text="Hauptkategorie", tickfont={"size": 8})
fig.update_xaxes(tickangle=45)
# add y axis label
fig.update_yaxes(title_text="Median time to resolution in hours")
# set width of plot
fig.update_layout(width=1000)
fig.show()

In [44]:
result = (
    df[df["Hauptkategorie"].isin(top15)][["Hauptkategorie", "resolution", "key"]]
    .groupby(["Hauptkategorie", "resolution"])
    .count()
    .reset_index()
)
# sort by overall count
result = result.sort_values("key", ascending=False)
# normalize per Hauptkategorie
result = result.groupby("Hauptkategorie").apply(
    lambda x: x.assign(Share=x["key"] / x["key"].sum())
)

fig = px.bar(result, x="Hauptkategorie", y="Share", color="resolution")
fig.update_layout(width=1000)
fig.show()

/var/folders/9v/qplwh6y13f93b4bjw7srn70r0000gn/T/ipykernel_64454/646240013.py:5: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [45]:
result = (
    df[["Hauptkategorie", "resolution", "key"]]
    .groupby(["Hauptkategorie", "resolution"])
    .count()
    .reset_index()
)
# sort by overall count
result = result.sort_values("key", ascending=False)

fig = px.bar(result, x="Hauptkategorie", y="key", color="resolution")
fig.update_layout(width=1000)
fig.show()

In [36]:
result

,Hauptkategorie,resolution,key
23,Computerkasse,Same day,405
91,System/Anlage,Same day,205
56,Lager,Same day,204
95,Textverarbeitung,Same day,189
8,Auftrag Allgemein,> 1 day,159
...,...,...,...
47,Intern,> 1 day,1
4,Anamnese,> 1 day,1
98,Weisse Preisliste,> 1 day,1
99,iSyncro,> 1 day,1


In [18]:
result = (
    df[df["currentstatus_name"] == "Fertig"][["Hauptkategorie", "key"]]
    .groupby("Hauptkategorie")
    .count()
    .reset_index()
    .sort_values("key", ascending=False)
)
result["Cumulative share"] = result["key"].cumsum() / result["key"].sum()
result["Share"] = result["key"] / result["key"].sum()


# Plot the main categories with a percentage bar in terms of count and include the cumulative count as a line
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar trace
fig.add_trace(
    go.Bar(
        x=result["Hauptkategorie"],
        y=result["Share"],
        name="Share",
    ),
    secondary_y=False,
)

# Line trace (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=result["Hauptkategorie"],
        y=result["Cumulative share"],
        mode="lines+markers",
        name="Cumulative share",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Distribution of tickets by main category",
    xaxis_title="Hauptkategorie",
    yaxis_title="Share of tickets",
)
fig.update_yaxes(title_text="Cumulative share", secondary_y=True)
# add x axis label in font size 8
fig.update_xaxes(title_text="Hauptkategorie", tickfont={"size": 9})
fig.update_xaxes(tickangle=45)
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
# set width of plot
fig.update_layout(width=1200)
fig.update_layout(height=500)
fig.show()

## Analyzing topics

In [47]:
print("Top 15 main categories:", list(top15))

Top 15 main categories: ['Computerkasse', 'System/Anlage', 'Textverarbeitung', 'Auftrag Allgemein', 'Lager', 'Terminkalender', 'Netzwerk/Internet', 'Paskal 3D', 'Datensicherung', 'Hardware - von uns', 'Datenservice/OLDS', 'Kundenkartei', 'KK-Abrechnung', 'Filialvernetzung']


### Get tickets for September-November per category

In [50]:
dff_oct2025 = df[(df["month"].isin([9, 10, 11])) & (df["Hauptkategorie"].isin(top15))]

In [52]:
dff_oct2025.shape

(1834, 29)

In [56]:
# Disabled: get_summary() lives in utils.py, which is not part of this
# repository. Restore utils.py to regenerate these LLM summaries.
# summaries = []
# for cat in top15:
#     descriptions = dff_oct2025[dff_oct2025["Hauptkategorie"] == cat]["description"]
#     titles = dff_oct2025[dff_oct2025["Hauptkategorie"] == cat]["summary"]
#     descriptions = [f"{t}\n{d}" for t, d in zip(titles, descriptions)]
#     summary = get_summary(descriptions)
#     summaries.append(f"## {cat}\n\n{summary}")
#
# summaries_markdown = "\n\n".join(summaries)
# # save summaries to file
# with open("summaries.md", "w") as f:
#     f.write(summaries_markdown)

In [ ]:
# Disabled: get_summary() lives in utils.py, which is not part of this
# repository. Restore utils.py to regenerate these LLM summaries.
# dff_oct2025_sameday = dff_oct2025[dff_oct2025["bdays"] == 0]
# summaries = []
# for cat in top15:
#     descriptions = dff_oct2025_sameday[dff_oct2025_sameday["Hauptkategorie"] == cat][
#         "description"
#     ]
#     titles = dff_oct2025_sameday[dff_oct2025_sameday["Hauptkategorie"] == cat][
#         "summary"
#     ]
#     descriptions = [f"{t}\n{d}" for t, d in zip(titles, descriptions)]
#     summary = get_summary(descriptions)
#     summaries.append(f"## {cat}\n\n{summary}")
#
# summaries_markdown = "\n\n".join(summaries)
# # save summaries to file
# with open("summaries_sameday.md", "w") as f:
#     f.write(summaries_markdown)

In [ ]:
# Disabled: get_summary() lives in utils.py, which is not part of this
# repository. Restore utils.py to regenerate these LLM summaries.
# (kept live - cell below still uses this frame)
dff_oct2025_emails = dff_oct2025[dff_oct2025["request_type"] == "Anfrage per E-Mail"]
# summaries = []
# for cat in top15:
#     descriptions = dff_oct2025_emails[dff_oct2025_emails["Hauptkategorie"] == cat][
#         "description"
#     ]
#     titles = dff_oct2025_emails[dff_oct2025_emails["Hauptkategorie"] == cat]["summary"]
#     descriptions = [f"{t}\n{d}" for t, d in zip(titles, descriptions)]
#     summary = get_summary(descriptions)
#     summaries.append(f"## {cat}\n\n{summary}")
#
# summaries_markdown = "\n\n".join(summaries)
# # save summaries to file
# with open("summaries_emails.md", "w") as f:
#     f.write(summaries_markdown)

'summaries = []\nfor cat in top15:\n    descriptions = dff_oct2025_emails[dff_oct2025_emails[\'Hauptkategorie\'] == cat][\'description\']\n    titles = dff_oct2025_emails[dff_oct2025_emails[\'Hauptkategorie\'] == cat][\'summary\']\n    descriptions = [f"{t}\n{d}" for t, d in zip(titles, descriptions)]\n    summary = get_summary(descriptions)\n    summaries.append(f"## {cat}\n\n{summary}")\n\nsummaries_markdown = "\n\n".join(summaries)\n# save summaries to file\nwith open(\'summaries_emails.md\', \'w\') as f:\n    f.write(summaries_markdown)'

In [64]:
dff_oct2025_emails

,key,summary,description,status,status_category,created,updated,labels,source,priority,...,time_to_resolution_days,resolution,bdays,created_string,updated_string,year,month,Hauptkategorie,Unterkategorie,time_to_resolution_bin
39,SDIPR-100008140,Button fehlt in Lawi-Auskunft 16404,-------- Weitergeleitete Nachricht --------\nB...,Fertig,Fertig,2025-11-28 16:21:05.558000+00:00,2025-11-28 16:24:24.088000+00:00,[],Anruf,Normal,...,0.0,Same day,0,2025-11-28,2025-11-28,2025,11,Lager,Auskunft,0–1
51,SDIPR-100008128,Fil 107 Etikettenformular muss wieder hinterle...,"Hallo Manu,\n\nMan muss sich auf dem Rechner a...",Fertig,Fertig,2025-11-28 13:29:19.508000+00:00,2025-12-01 10:18:06.683000+00:00,[],Anruf,Normal,...,2.0,> 1 day,1,2025-11-28,2025-12-01,2025,11,Lager,NaN,48–72
52,SDIPR-100008127,Fil 97 Etikettenformular muss wieder hinterleg...,"Hallo Manu,\n\nMan muss sich auf dem Rechner a...",Warten auf Support,Zu erledigen,2025-11-28 13:26:49.204000+00:00,2025-11-28 13:28:20.104000+00:00,[],Anruf,Normal,...,0.0,Same day,0,2025-11-28,2025-11-28,2025,11,Lager,NaN,NaN
53,SDIPR-100008126,Fil 65 Etikettenformular muss wieder hinterleg...,"Hallo Manu,\n\nMan muss sich auf dem Rechner a...",Warten auf Support,Zu erledigen,2025-11-28 13:21:17.793000+00:00,2025-11-28 13:25:43.505000+00:00,[],Anruf,Normal,...,0.0,Same day,0,2025-11-28,2025-11-28,2025,11,Lager,Etikettendruck,NaN
54,SDIPR-100008125,Fil 60 Etikettenformular muss wieder hinterleg...,"Hallo Manu,\n\nMan muss sich auf dem Rechner a...",Warten auf Support,Zu erledigen,2025-11-28 13:18:07.834000+00:00,2025-11-28 13:19:58.851000+00:00,[],Anruf,Normal,...,0.0,Same day,0,2025-11-28,2025-11-28,2025,11,Lager,Etikettendruck,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3170,SDIPR-100004956,Monatsabschluss exportieren,------- Weitergeleitete Nachricht --------\n\n...,Fertig,Fertig,2025-09-01 10:16:03.475000+00:00,2025-09-23 10:47:23.831000+00:00,[],Anruf,Sehr Hoch,...,22.0,> 1 day,16,2025-09-01,2025-09-23,2025,9,Auftrag Allgemein,NaN,NaN
3173,SDIPR-100004953,Fehlerhafte KK Abrechnung AOK,None,Fertig,Fertig,2025-09-01 09:57:50.088000+00:00,2025-11-04 10:06:40.904000+00:00,[],Anruf,Sehr Hoch,...,64.0,> 1 day,46,2025-09-01,2025-11-04,2025,9,KK-Abrechnung,NaN,NaN
3174,SDIPR-100004952,Dateifehler Absender 320105698 Annahmestelle 6...,-------- Weitergeleitete Nachricht --------\nB...,Fertig,Fertig,2025-09-01 09:49:31.830000+00:00,2025-09-09 15:08:29.824000+00:00,[],Anruf,Sehr Hoch,...,8.0,> 1 day,6,2025-09-01,2025-09-09,2025,9,KK-Abrechnung,NaN,168–336
3177,SDIPR-100004949,Fehlerhafte KK Abrechnung,-------- Weitergeleitete Nachricht --------\n\...,Fertig,Fertig,2025-09-01 09:32:48.391000+00:00,2025-09-09 15:08:57.644000+00:00,[],Anruf,Sehr Hoch,...,8.0,> 1 day,6,2025-09-01,2025-09-09,2025,9,KK-Abrechnung,NaN,168–336


## Focus on E-Mail tickets

In [ ]:
dfem = df[df["request_type"] == "Anfrage per E-Mail"]

In [61]:
dfem.shape

(0, 29)

## Clone relationships

In [22]:
def get_clone_relations(issue):
    cloned_from = None
    clones = []

    for link in issue.fields.issuelinks:
        t = link.type

        # this issue was cloned FROM another
        if hasattr(link, "outwardIssue") and t.name == "Cloners":
            cloned_from = link.outwardIssue.key

        # this issue was cloned BY another
        if hasattr(link, "inwardIssue") and t.name == "Cloners":
            clones.append(link.inwardIssue.key)

    return cloned_from, clones

In [12]:
issues2 = issues[:50]
for issue in issues2:
    issue = jira.issue(issue.key)
    cloned_from, clones = get_clone_relations(issue)

    if cloned_from is not None and cloned_from not in clones:
        print(issue.key, issue.fields.summary, "Cloned from:", cloned_from)
    if len(clones) > 0:
        print(issue.key, issue.fields.summary, "Clones:", clones)